# Background Theory — Sports-Betting Market Modeling
### (Point Spread Prediction & Similar Projects)

This notebook explains the *why* behind every technique used in the project, with special attention to the concepts that are specific to betting-market data (they don't come up in a typical tabular-regression tutorial). Small illustrative code cells use synthetic data — they demonstrate the concept, not the actual project dataset.


## 1. What a Point Spread Actually Is

A **point spread** is a bookmaker's estimate of the expected margin of victory, expressed so that a bet on either side carries roughly even odds. If the home team is favored by 4.5 points, a bettor who takes the home team only "wins" if the home team's actual margin of victory exceeds 4.5; a bettor who takes the away team wins if the home team wins by less than 4.5 or loses outright ("covering" and "not covering" the spread). Spreads exist specifically to balance betting volume on both sides, which is why they move in response to money flow, not just in response to new information about the two teams.

**Opening vs. closing lines.** The *opening* line is a bookmaker's or market's initial number, typically set from statistical models and expert judgment. The *closing* line is the final number right before the event, having absorbed all the betting activity (and any last-minute information — injuries, weather, lineup news) that came in between. In efficient betting markets, the closing line is generally considered the more accurate estimate, because it reflects the aggregated information and money of the entire betting public plus professional ("sharp") bettors.

**Moneyline vs. spread.** A moneyline directly quotes the payout odds for each team to win outright (no margin involved), while a spread handicaps the favorite by points. Both are just different notations for essentially the same underlying assessment of each team's win probability and expected margin — which is exactly why using one to predict the other is a form of leakage (Section 3).


## 2. Why This Domain Needs Domain-Specific Feature Engineering

**Differentials, not raw levels.** A spread is a statement about the *gap* between two teams' expected performance, not about either team's absolute level. A model given `home_ppg=115, away_ppg=110` and one given `home_ppg=95, away_ppg=90` should produce similar-shaped predictions (both a 5-point scoring edge), which is much easier for a model to learn if you hand it the differential (`home_ppg - away_ppg = 5` in both cases) directly rather than making it discover subtraction from two separate raw columns.

**"Smart money" vs. "public money."** Betting markets have a well-documented dynamic where casual/recreational bettors ("the public") tend to bet on favorites, popular teams, and the "obvious" side of a game in disproportionate volume, while professional bettors ("sharps") bet more selectively and are believed, on average, to be better-informed. A gap between the percentage of *bets* on a side and the percentage of *money* on that side (since sharps tend to bet larger amounts) — or, more simply, a gap between overall public betting percentage and known-sharp betting percentage — is a classic signal that the closing line may still move even after a given point, because sportsbooks often adjust more in response to sharp money than to bet volume alone.

**Situational factors compound rather than replace team strength.** Rest, injuries, and travel don't usually overturn a large talent gap, but they reliably shift a line by a point or two — which is why they're typically modeled as additive adjustments to a baseline team-strength estimate, not as standalone predictors on their own.


## 3. Leakage Is a Bigger, Sneakier Risk in Betting-Market Data

This is the concept most worth internalizing from this project, because it doesn't show up as clearly in most tabular-ML tutorials.

**The general leakage rule** (see also the Laptop Price project's Background Theory notebook): a feature leaks information if it was computed using information that wouldn't actually be available, or if it's improperly derived using data from what should be held-out ("test") observations.

**The betting-market-specific version of this problem:** even a feature that *is* technically available before your target value exists can still defeat the purpose of your model if it's just another representation of the same market assessment you're trying to predict or improve on.

- If your goal is **"predict the closing line as accurately as possible, using any legitimate signal"** — using the *opening* line as a feature is perfectly legitimate; it's known before the close and genuinely informative.
- If your goal is **"build an independent rating model to see whether it can find value the market hasn't fully priced in"** — using the opening line as a feature undermines the entire exercise, because you'd mostly be learning "the market's number, slightly adjusted" rather than demonstrating your model has independent value.

**The moneyline/spread relationship is even more direct leakage** in almost any framing, because they're mathematically near-equivalent representations of the same closing assessment — closer to using a target's own transformed value as a feature (like trying to predict a temperature in Celsius using a column that's the same temperature in Fahrenheit) than to using a genuinely independent signal.

**Practical rule of thumb:** before including any "market" column as a feature, ask *"is this describing the underlying event (teams, players, situational factors), or is this describing what someone (the market, another bettor, an algorithm) already concluded about the event?"* The former is safe; the latter needs careful, goal-specific justification.


In [1]:
import numpy as np
import pandas as pd

# Demonstration: a feature that's a noisy version of the target massively inflates apparent R^2
rng = np.random.default_rng(0)
n = 300
true_signal = rng.normal(0, 10, n)
target = true_signal + rng.normal(0, 3, n)          # what we're predicting
market_quote = target + rng.normal(0, 0.5, n)        # a "different format" of nearly the same number

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

honest_features = pd.DataFrame({'true_signal': true_signal})
leaky_features = pd.DataFrame({'true_signal': true_signal, 'market_quote': market_quote})

honest_r2 = r2_score(target, LinearRegression().fit(honest_features, target).predict(honest_features))
leaky_r2 = r2_score(target, LinearRegression().fit(leaky_features, target).predict(leaky_features))
print(f"Honest R2 (real signal only): {honest_r2:.3f}")
print(f"Leaky R2 (market's own near-copy of the target included): {leaky_r2:.3f}")


Honest R2 (real signal only): 0.925
Leaky R2 (market's own near-copy of the target included): 0.998


The leaky version's R² jumps close to 1 — not because the model learned anything new about the underlying event, but because it was handed a slightly-noisy copy of the answer.

## 4. Additive vs. Interaction-Heavy Relationships (and Why Model Choice Should Follow the Data)

**Additive relationships** are ones where each feature's effect adds onto the others independently of their values (e.g. `spread ≈ a·net_rating_diff + b·rest_advantage + c·injury_advantage + constant`). Linear regression is *exactly* the right tool for this shape — it directly estimates each additive coefficient.

**Interaction-heavy relationships** are ones where a feature's effect depends on the value of another feature (e.g. RAM matters much more for price *if* the laptop is also a premium brand than if it's a budget brand). Tree ensembles naturally represent this kind of relationship because each split can be conditioned on the outcome of previous splits.

**The practical lesson from comparing this project to the laptop-price project:** there is no universally "better" algorithm. Random Forests and Gradient Boosting are excellent defaults *because* real-world tabular data is often at least somewhat interaction-heavy — but when a relationship is genuinely close to additive (as constructed point spreads tend to be, since oddsmakers themselves often build spreads from additive power-rating systems), a simple linear model can match or beat a more "sophisticated" ensemble, train faster, and be far easier to interpret via its coefficients. **Always run the comparison — don't assume the answer.**


## 5. Evaluating a Model Meant to Compete With a Market

**MAE in point-spread units is directly interpretable** ("off by 1.9 points on average") in a way that translates naturally to the domain — unlike, say, an abstract accuracy percentage.

**R² has a natural benchmark here that most tabular projects lack: the market's own accuracy.** In a mature betting market, the closing line is typically a very strong predictor of the actual outcome margin (though far from perfect — sports outcomes have enormous game-to-game variance). A model's R² should be interpreted relative to how good the *market itself* would score against actual outcomes, not just relative to zero.

**MAPE is often close to unusable for spread data specifically**, because the target frequently takes values near zero (a "pick'em" game, spread ≈ 0) — dividing by a near-zero true value makes the percentage error explode or become undefined for exactly the closest, most competitive games. Prefer MAE/RMSE in points for this kind of target, and mention MAPE's instability explicitly rather than silently reporting a possibly-meaningless number.


## 6. Backtesting a Decision Rule (Not Just Scoring a Model)

A backtest asks a different question than a standard train/test evaluation: not *"how accurate is the model on average?"* but *"if I had used this model's output to make a specific decision (place a bet, flag a trade, approve a loan), how would that decision have performed against real outcomes?"*

**Key backtesting principles, illustrated by this project's Section 13:**

- **Use genuinely held-out outcomes.** `home_margin` was never in the feature set — it's brought back in only after predictions are made, purely to score the *decisions*, which is what makes this a legitimate backtest rather than another leakage problem.
- **Define the decision rule explicitly and in advance** (e.g. "bet when the model disagrees with the market by ≥2 points"), rather than tuning the threshold after seeing which one produces the best-looking result — the latter is a form of overfitting called **backtest overfitting** or "p-hacking a trading strategy."
- **Compare against an explicit, meaningful baseline** — here, both "bet every game" and the mathematically-derived breakeven win rate given the cost of betting (-110 odds require winning 52.4% of the time just to break even, because of the bookmaker's built-in margin, the "vig" or "juice").
- **Distrust small, favorable-looking backtests.** A strategy that wins 60% of 88 bets could easily be within the range of normal statistical variation even with zero true underlying edge; a rule of thumb from quantitative finance is that a backtest needs many hundreds to thousands of independent trials, ideally across multiple time periods/market conditions, before its results should meaningfully update your confidence in a real edge.
- **Simulated/synthetic-data backtests demonstrate methodology, not real-world validity.** This is worth restating: nothing about this notebook's specific ROI numbers should be treated as evidence about real sports-betting markets, which are far more competitive and information-efficient than a small synthetic dataset.


## 7. Model Persistence for a Model That Will Be Run Repeatedly

A production spread model would typically be re-run every day (or several times a day) as new team-performance data, injury news, and betting percentages come in. This makes the inference-function discipline from Section 14 of the Solutions notebook especially important: every one of the raw-column cleaning and feature-engineering steps must be captured in a callable function (not just performed inline in a notebook), because a deployed system needs to apply *identical* transformations to fresh, no-target-yet data on a recurring schedule, not just once for a one-off backtest.


## 8. How This Generalizes Beyond NBA Spreads

The same shape — audit → clean/engineer differentials → EDA with a market-specific leakage audit → leakage-safe encode → compare model families → tune/cross-validate → evaluate with domain-appropriate metrics → backtest a decision rule → persist for repeated use — applies to:

- **Other sports' spreads/totals** (NFL, soccer goal-line/handicap markets, tennis game-spread markets): the same additive-power-rating intuition and opening/closing-line leakage trap apply almost unchanged.
- **Financial market prediction** (predicting whether a security's price will move relative to what's already priced in): the exact same "don't feed the market's own current price back in as if it were independent information" leakage trap appears constantly in quantitative finance.
- **Insurance/actuarial pricing**: predicting a "fair" premium from risk factors, then comparing it to a competitor's quoted premium to find underpriced/overpriced policies — structurally the same "build an independent estimate, then diff against a market quote" project shape.
- **Any prediction market or forecasting competition** (election forecasting, prediction markets like sports-adjacent futures): the recurring lesson is the same — a market price already aggregates a huge amount of information, so an "independent" model needs to be evaluated relative to how much it adds *beyond* that price, not evaluated in isolation.

What changes across domains: the specific situational features and the market-quote columns to watch out for. What stays constant: differential feature construction where the target is fundamentally about a gap between two things, rigorous leakage-checking specific to any "market quote" columns, comparing model families rather than assuming complexity wins, and backtesting decision rules against genuine held-out outcomes with an explicit, pre-registered rule and a meaningful baseline.
